In [1]:
import numpy as np
from scipy.spatial import cKDTree

In [2]:
training_parameters = np.load('CosmoGrid_params.npz').files
param_names = ['Om', 'sigma8', 'w0', 'ns', 'Ob', 'H0', 'bary_Mc', 'bary_nu']
training_parameters = np.array([np.load('CosmoGrid_params.npz')[par][:1250] for par in param_names]).T

In [15]:
(training_parameters - training_parameters.min(axis=0))/(training_parameters.max(axis=0) - training_parameters.min(axis=0))

array([[0.49963334, 0.50122429, 0.47991866, ..., 0.50024438, 0.5       ,
        0.50024438],
       [0.74993889, 0.25048972, 0.21962379, ..., 0.25      , 0.75024438,
        0.25      ],
       [0.24932779, 0.75195886, 0.74021352, ..., 0.75048876, 0.24975562,
        0.75048876],
       ...,
       [0.24663896, 0.44319295, 0.82079309, ..., 0.08553275, 0.32575758,
        0.09921799],
       [0.99755561, 0.19245837, 0.56049822, ..., 0.33577713, 0.57600196,
        0.34946237],
       [0.62209729, 0.31782566, 0.43035079, ..., 0.46089932, 0.70112414,
        0.97507331]])

In [4]:
tree = cKDTree(training_parameters)
dists, _ = tree.query(training_parameters, k=2)  # k=2: first is self, second is NN
min_spacing = np.min(dists[:, 1])

In [7]:
min_spacing

0.26913812968898043

In [ ]:
def generate_deterministic_perturbations(grid_nodes, n_realizations=7, frac=0.2, seed=42):
    """
    Generate deterministic parameter perturbations for each grid node.
    
    Parameters
    ----------
    grid_nodes : np.ndarray
        Array of shape (n_nodes, d) with the parameter values of the grid.
    n_realizations : int
        Number of realizations per node.
    frac : float
        Fraction of the minimum grid spacing to use as perturbation scale.
    seed : int
        Global random seed for reproducibility.
    
    Returns
    -------
    perturbed_params : np.ndarray
        Array of shape (n_nodes, n_realizations, d) with perturbed parameters.
    """

    n_nodes, d = grid_nodes.shape
    
    # 1. Normalize each parameter to [0,1]
    min_vals = grid_nodes.min(axis=0)
    max_vals = grid_nodes.max(axis=0)
    grid_norm = (grid_nodes - min_vals) / (max_vals - min_vals)

    # 2. Compute nearest neighbor distances in normalized space
    tree = cKDTree(grid_norm)
    dists, _ = tree.query(grid_norm, k=2)
    min_spacing = np.min(dists[:, 1])

    perturbed_norm = np.zeros((n_nodes, n_realizations, d))
    
    base_rng = np.random.RandomState(seed)

    # Create a base RNG
    base_rng = np.random.RandomState(seed)

    
    